# Chapter 2 -- The Agent Loop from Scratch (Ollama Local)

Work through this notebook **after reading** `notes/ch02-agent-loop-from-scratch.md`. This version of the chapter builds a real agent loop against the industry standard **OpenAI tool-use wire format** (`tool_calls`, `finish_reason: "tool_calls"`) -- the exact mechanics the notes describe in detail, using notes Section 11's own `read_file` / `word_count` task.

Three exercises below have a stub to fill in: the **budget guard**, **parallel tool-call handling**, and **JSONL trajectory logging**. Each is verified directly below it against your real local Ollama model -- there is no scripted fake client here. Because a real model's answer isn't fixed run to run, what's checked is structural (did your code do the right thing with whatever the model actually said), not an exact expected trace.

## One-time local setup

This notebook needs a local Ollama server, not an API key -- everything below runs
entirely on your own machine, for free.

**Install Ollama:**

- macOS: `brew install ollama` (already covered if you followed this repo's other setup notes) -- or the installer at [ollama.com/download](https://ollama.com/download)
- Windows: download and run the installer from [ollama.com/download](https://ollama.com/download). It installs Ollama as a background service, so there is no separate `ollama serve` step to run by hand.
- Linux / Ubuntu: `curl -fsSL https://ollama.com/install.sh | sh`

**Start the server** (macOS/Linux only -- the Windows installer already runs this as a service), left running in its own terminal:
```
ollama serve
```

**Pull a model that supports tool calling** -- an 8B-class model is the sweet spot on
a 16GB-unified-memory laptop: big enough to follow this chapter's two-tool task
reliably, small enough to leave headroom for everything else you have open.
This notebook is set up for two already-pulled options (check with `ollama list`):
`llama3.1:latest` and `qwen3:8b`. If you don't have one yet: `ollama pull llama3.1` or `ollama pull qwen3:8b`.

To switch which one the notebook actually calls, edit `OLLAMA_MODEL` in the cell
below -- comment out the active line, uncomment the other. If you're on
different hardware with different models pulled, add a third constant the same
way; nothing else in this notebook needs to change.

In [ ]:
%pip install openai

import os
from pathlib import Path
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Models pulled locally (see `ollama list`). Swap the active model by moving
# which line is commented out -- everything else in this notebook stays the same.
OLLAMA_MODEL_LLAMA = "llama3.1:latest"
OLLAMA_MODEL_QWEN3 = "qwen3:8b"

# OLLAMA_MODEL = OLLAMA_MODEL_LLAMA
OLLAMA_MODEL = OLLAMA_MODEL_QWEN3

# We use the standard OpenAI SDK, but point it to the local Ollama server.
# This means your code is 100% compatible with real OpenAI models if you just
# swap the base_url and use a real API key later.
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


def test_connection(client, model_name):
    print(f"Testing connection to local Ollama (model={model_name})...")
    try:
        response = client.chat.completions.create(
            model=model_name, max_tokens=50,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = response.choices[0].message.content
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("Every exercise below calls this same client directly -- without Ollama")
        print("running, Exercises 2 and 3 will fail to connect (Exercise 1's budget-guard")
        print("check never calls the model at all, so it still works either way).")
        print(f"Make sure 'ollama serve' is running and you have pulled: ollama pull {model_name}")


test_connection(client, OLLAMA_MODEL)

## The task and its two tools

The agent's job: read two text files and report their combined word count. Two tools are available -- `read_file(path)` and `word_count(text)` -- the exact pair from notes Section 11's dry-run.

In [ ]:
import json

SAMPLE_DIR = Path.cwd() / "ch02_sample_files"
SAMPLE_DIR.mkdir(exist_ok=True)

(SAMPLE_DIR / "report.txt").write_text(
    "Quarterly revenue grew twelve percent driven by strong demand in the "
    "enterprise segment and continued expansion in international markets."
)
(SAMPLE_DIR / "business.txt").write_text(
    "The board approved a new capital allocation plan focused on research "
    "and disciplined cost management across every division this year."
)

print("Sample files created:")
for f in sorted(SAMPLE_DIR.iterdir()):
    print(f"  {f.name}: {len(f.read_text().split())} words")


def read_file(path: str) -> str:
    """Return the contents of a file at `path` (resolved inside the sample directory)."""
    target = SAMPLE_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"{path} not found in {SAMPLE_DIR.name}/")
    return target.read_text()


def word_count(text: str) -> str:
    """Return the number of whitespace-separated words in `text`, as a string."""
    return str(len(text.split()))


TOOL_DISPATCH = {
    "read_file": read_file,
    "word_count": word_count,
}

# In the OpenAI/Ollama schema, tools are defined like this:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of a text file by name. Call this before word_count if you need the file content.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "File name, e.g. 'report.txt'"}},
                "required": ["path"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "word_count",
            "description": "Count the number of whitespace-separated words in a string of text you already have.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string", "description": "The text to count words in"}},
                "required": ["text"],
            }
        }
    },
]

# Real answer to compare every run's final answer against -- computed once,
# directly, with no model involved.
_COMBINED_COUNT = len(read_file("report.txt").split()) + len(read_file("business.txt").split())
print(f"Real combined word count (for later comparison): {_COMBINED_COUNT}")

## Same loop, different wire format: OpenAI/Ollama vs Anthropic

The notes chapter teaches Anthropic's Messages API shape end to end: `tool_use` /
`tool_result` blocks, `stop_reason`, and -- per notes Section 7 -- **all** tool
results for one turn packed into a single `role: "user"` message. Ollama's
OpenAI-compatible server speaks a different, but structurally equivalent, dialect:

| Concept | Anthropic (notes' main format) | OpenAI / Ollama (this notebook) |
|---|---|---|
| "the model wants to act" | `stop_reason == "tool_use"` | `finish_reason == "tool_calls"` |
| the request(s) | one or more `tool_use` blocks inside one assistant message | one or more `tool_calls` entries inside one assistant message |
| the answer(s) | **all** results packed into ONE `role: "user"` message | **one separate `role: "tool"` message per call**, each carrying its own id |
| matching id field | `tool_use_id` | `tool_call_id` |
| "the model is done" | `stop_reason == "end_turn"` | `finish_reason == "stop"` |

The row worth sitting with is the third one. Anthropic batches every result from one
turn into a single user turn -- notes Section 7's entire point is that splitting
them "silently trains the model to stop batching." OpenAI's schema expects the exact
opposite: **one `tool` message per call, never combined into one.** Both are
internally consistent designs for the same underlying idea -- the model asked for N
things, you answer N things -- they simply disagree on whether "one turn's worth of
answers" is one message or N messages. Get this backwards on either provider and the
request is rejected or silently degrades future tool-calling behavior; there is no
universal rule here except "match whichever schema you are actually talking to.".

In [ ]:
def execute_tool_call(tool_call) -> tuple:
    """
    Run one tool_call block through TOOL_DISPATCH.
    Returns (content_string, is_error) -- matches notes Section 6 exactly:
    exceptions become an error tool message, never a crash.
    """
    name = tool_call.function.name
    fn = TOOL_DISPATCH.get(name)
    if fn is None:
        return f"Error: no such tool '{name}'", True
    try:
        # OpenAI arguments are passed as a JSON string
        args = json.loads(tool_call.function.arguments)
        result = fn(**args)
        return str(result), False
    except Exception as exc:
        return f"Error: {exc}", True

## Exercises -- complete `run_agent_loop`

One function, three TODOs, matching notes Sections 5, 7, and 9:

1. **TODO 1 -- budget guard.** Before each request, if `cumulative_tokens` already exceeds `max_token_budget`, print a message and return early as `(None, step - 1, cumulative_tokens)`.
2. **TODO 2 -- parallel tool handling.** `tool_calls` may hold more than one block. In the OpenAI schema, you must append **ONE new message per tool call**, with `role: "tool"`, the `tool_call_id`, and the `content`.
3. **TODO 3 -- trajectory logging.** If `log_path` is given, append one JSON line per step (every step, including the final stop) recording `step`, `finish_reason`, `cumulative_tokens`, and `tool_calls_this_step`.

Each exercise is checked directly against your real local Ollama model, right below it -- see the flow diagram immediately below for how the whole function fits together before you fill anything in.

## How `run_agent_loop` Flows

This is exactly notes Section 3's "send -> read the stop signal -> branch ->
execute + append -> repeat" cycle, with OpenAI/Ollama's field names in place of
Anthropic's (see the wire-format contrast cell above). One pass through the `for`
loop is one "turn":

```
                    ┌───────────────────────────────────────────────┐
                    │                                                │
                    ▼                                                │
        ┌─────────────────────────┐                                 │
        │ TODO 1 -- budget guard  │  cumulative_tokens > budget?     │
        │ (checked BEFORE the     │──── yes ──▶ return (None, ...)   │
        │  next call is made)     │                                  │
        └────────────┬────────────┘                                  │
                      │ no                                            │
                      ▼                                                │
        ┌───────────────────────────────┐                              │
        │ client.chat.completions       │  POST to Ollama's            │
        │   .create(model, tools, msgs) │  OpenAI-compatible /v1 API   │
        └────────────┬──────────────────┘                              │
                      ▼                                                  │
        ┌─────────────────────────┐                                     │
        │ TODO 3 -- log this step │  every step, whatever finish_reason │
        │ to trajectory_ollama    │  turns out to be                    │
        │ .jsonl                  │                                      │
        └────────────┬────────────┘                                      │
                      ▼                                                    │
        ┌─────────────────────────────────┐                                │
        │ append the assistant turn to    │  full message, tool_calls     │
        │ `messages` (role="assistant")   │  and all -- BEFORE acting     │
        └────────────┬─────────────────────┘                              │
           ┌──────────┴───────────┐                                        │
           ▼                      ▼                                        │
   finish_reason ==        finish_reason ==                                │
     "stop"                  "tool_calls"                                  │
           │                      │                                        │
           ▼                      ▼                                        │
   return final_text    ┌──────────────────────────────┐                   │
                         │ TODO 2 -- for EACH tool_call, │                  │
                         │ execute it, then append ONE   │                  │
                         │ role="tool" message with the  │                  │
                         │ matching tool_call_id          │                 │
                         └──────────────┬─────────────────┘                 │
                                        └─────────────────────────────────────┘
                                             loop back to top (step += 1)
```

Concretely, here is what a *correct* implementation's first two turns look like on
this chapter's task (idealized -- a real small model won't always follow this
exactly, which is what the verification cells below are for):

```
Step 1 -- call 1:
  messages going in:  [ {role: "user", content: "Read report.txt and business.txt..."} ]
  model responds: finish_reason="tool_calls", two tool_calls --
      read_file(path="report.txt"), read_file(path="business.txt")
  TODO 1 check: cumulative_tokens (0) > budget?  No -> the call above was allowed to happen.
  TODO 3 logs {"step": 1, "finish_reason": "tool_calls", "tool_calls_this_step": 2, ...}
  the assistant turn is appended (role="assistant", tool_calls=[...])
  TODO 2: each tool is executed, and ONE role="tool" message is appended per call --
      `messages` is now 4 entries long: user, assistant, tool, tool

Step 2 -- call 2:
  messages going in: all 4 entries above, resent in full (the API is stateless)
  model responds: finish_reason="stop", content="The combined word count is ..."
  TODO 3 logs step 2; the assistant turn is appended; the function returns
      (final_text, 2, cumulative_tokens)
```

Nothing above is unique to this specific run -- it is the same shape every turn
takes, whether the model needs 2 steps or 6, and whether it makes 1 tool call or 4
in a single turn.

In [ ]:
import json

MAX_TOKEN_BUDGET = 5000
MAX_STEPS = 10


def run_agent_loop(client, model, tools, messages, max_steps=MAX_STEPS,
                    max_token_budget=MAX_TOKEN_BUDGET, log_path=None):
    """
    The full agent loop. Requests completions, branches on finish_reason,
    executes tools, guards against runaway budgets, and logs every step.
    Returns: (final_text_or_None, steps_taken, cumulative_tokens)
    """
    cumulative_tokens = 0
    if log_path is not None and log_path.exists():
        log_path.unlink()  # start each run with a clean trajectory file

    for step in range(1, max_steps + 1):
        # --- TODO 1: BUDGET GUARD ---
        # If cumulative_tokens already exceeds max_token_budget, print a
        # clear message and return (None, step - 1, cumulative_tokens).
        pass  # TODO 1

        response = client.chat.completions.create(
            model=model, tools=tools, messages=messages,
        )
        cumulative_tokens += response.usage.prompt_tokens + response.usage.completion_tokens

        finish_reason = response.choices[0].finish_reason
        msg = response.choices[0].message

        print(f"STEP {step} | finish_reason={finish_reason} | "
              f"cumulative_tokens={cumulative_tokens}")

        # --- TODO 3: TRAJECTORY LOGGING (every step, regardless of stop_reason) ---
        # If log_path is not None, append one JSON line recording: step,
        # finish_reason, cumulative_tokens, and the number of tool_calls this step.
        pass  # TODO 3

        # In OpenAI, the assistant message itself must be appended back to history!
        assistant_msg = {"role": "assistant"}
        if msg.content: assistant_msg["content"] = msg.content
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": tc.type, "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if finish_reason == "stop":
            print(f"  Final answer: {msg.content}")
            return msg.content, step, cumulative_tokens

        # Execute parallel tools
        if finish_reason == "tool_calls":
            for tool_call in msg.tool_calls:
                content_str, is_error = execute_tool_call(tool_call)
                status = "ERROR" if is_error else "ok"
                print(f"    tool_use {tool_call.function.name}({tool_call.function.arguments}) -> [{status}] "
                      f"{content_str[:60]}")

                # --- TODO 2: APPEND TOOL RESULTS ---
                # In OpenAI format, append ONE message dict per tool call.
                # Format: {"role": "tool", "tool_call_id": tool_call.id, "content": content_str}
                pass # TODO 2

    print(f"Hit max_steps ({max_steps}) without a natural stop.")
    return None, max_steps, cumulative_tokens

**Verification -- Exercise 1 (budget guard)**

Run with an *impossible* budget (`-1` -- nothing can ever be lower than the 0
tokens spent before the very first call) so the pre-flight check is guaranteed to
trip before any real API call is made. This is fully deterministic even against a
real, non-deterministic model, because it never actually reaches the network.

In [ ]:
fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count."}]
final_text, steps_taken, tokens = run_agent_loop(
    client, model=OLLAMA_MODEL, tools=TOOL_SCHEMAS,
    messages=fresh_messages, max_token_budget=-1,  # impossible to satisfy -- guaranteed instant trip
)

assert final_text is None, "Expected the budget guard to stop the loop before any call was made."
assert steps_taken == 0, f"Expected zero steps taken (guard fires before step 1 even starts), got {steps_taken}."
assert tokens == 0, f"Expected zero tokens spent (no API call should have happened), got {tokens}."
print(f"PASS -- budget guard stopped the loop before any real API call was made ({tokens} tokens spent).")

**Verification -- Exercise 2 (parallel tool handling)**

Run the real task against your real model and check a structural invariant that
must hold no matter what the model actually decided to call: every `tool_calls`
turn must be followed by exactly one `role: "tool"` message per call, each
carrying the matching `tool_call_id`. This holds whether the model batches both
`read_file` calls into one turn or splits them across two -- both are legal, and
neither should make this check fail.

In [ ]:
fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count. Use the tools -- do not guess."}]
try:
    final_text, steps_taken, tokens = run_agent_loop(
        client, model=OLLAMA_MODEL, tools=TOOL_SCHEMAS,
        messages=fresh_messages, max_token_budget=20_000,
    )
except Exception as exc:
    raise AssertionError(
        f"run_agent_loop raised {type(exc).__name__}: {exc} -- likely TODO 2 not filled in yet "
        "(an unresolved tool_calls turn makes the next request invalid)."
    ) from exc

tool_turns = [i for i, m in enumerate(fresh_messages) if m.get("role") == "assistant" and m.get("tool_calls")]
if not tool_turns:
    print("NOTE: the real model didn't call any tools this run (it guessed instead of using them).")
    print("Nothing to verify against -- just re-run this cell; local models occasionally skip tools.")
else:
    for i in tool_turns:
        expected_ids = [tc["id"] for tc in fresh_messages[i]["tool_calls"]]
        actual = fresh_messages[i + 1: i + 1 + len(expected_ids)]
        assert len(actual) == len(expected_ids), (
            f"Turn {i} made {len(expected_ids)} tool call(s) but only {len(actual)} "
            f"role='tool' message(s) followed -- check TODO 2."
        )
        for result_msg, expected_id in zip(actual, expected_ids):
            assert result_msg.get("role") == "tool", f"Expected role='tool', got {result_msg.get('role')!r}."
            assert result_msg.get("tool_call_id") == expected_id, "tool_call_id mismatch -- results out of order?"
    print("PASS -- every tool call the real model made got exactly one matching tool-result message back.")

print(f"Real run: {steps_taken} step(s), {tokens} cumulative tokens.")
print(f"Real combined word count for comparison: {_COMBINED_COUNT}; model said: {final_text!r}")

**Verification -- Exercise 3 (trajectory logging)**

Run once more against the real model with `log_path` set, and confirm every step
got written to disk as valid JSON -- this is checkable independent of what the
model actually said, since it only depends on TODO 3 firing once per step.

In [ ]:
TRAJECTORY_PATH = Path.cwd() / "trajectory_ollama.jsonl"

fresh_messages = [{"role": "user", "content": "Read report.txt and business.txt, then tell me the combined word count."}]
try:
    final_text, steps_taken, tokens = run_agent_loop(
        client, model=OLLAMA_MODEL, tools=TOOL_SCHEMAS,
        messages=fresh_messages, max_token_budget=20_000, log_path=TRAJECTORY_PATH,
    )
except Exception as exc:
    raise AssertionError(
        f"run_agent_loop raised {type(exc).__name__}: {exc} -- likely TODO 2 not filled in yet."
    ) from exc

assert TRAJECTORY_PATH.exists(), "No trajectory_ollama.jsonl was written -- check TODO 3."
lines = TRAJECTORY_PATH.read_text().strip().splitlines()
assert len(lines) == steps_taken, f"Expected {steps_taken} logged steps, found {len(lines)}."

for line in lines:
    record = json.loads(line)  # must parse as valid JSON
    assert {"step", "finish_reason", "cumulative_tokens", "tool_calls_this_step"} <= record.keys(), (
        f"Missing expected keys in logged record: {record}"
    )

print(f"PASS -- {len(lines)} step(s) logged to {TRAJECTORY_PATH.name}, each record well-formed.")
print(f"Real combined word count for comparison: {_COMBINED_COUNT}; model said: {final_text!r}")
for line in lines:
    print(" ", line)

## Key Takeaways

You now have a real, self-correcting, parallel-tool-aware, budget-guarded, self-logging agent loop -- built from a request/response cycle, a `finish_reason` branch, and a dispatch table using the **OpenAI tool calling schema** against your real local model.